# Reusable Template — Economic/Panel-Data Regression Pipeline

A dataset-agnostic notebook for **"predict an economic/financial outcome from macro or entity-level indicators, with accounting-style formatting and a forecast/quote column to audit for leakage"** projects: country panels, corporate financial-statement analysis, regional economic indicators, and similar problems.

**How to reuse this notebook:**
1. Fill in `CONFIG` (Section 1) for your dataset.
2. Fill in `clean_raw_columns()` (Section 3) — the accounting-percentage parser and whitespace/casing cleanup are provided generically; add your dataset's specific columns.
3. Fill in `STRUCTURAL_MISSING_COLS` and `engineer_domain_features()` (Section 4) for structurally-absent columns and any interaction terms domain knowledge suggests.
4. Review the leakage audit in Section 6, reasoning about correlation strength AND what each flagged column represents.


## 1. Configuration

In [ ]:
CONFIG = {
    'data_path': 'economy_indicators.csv',   # <-- change per project
    'target_col': 'gdp_growth_rate_pct',      # <-- change per project

    # Columns that summarize an outside party's own conclusion about the same/similar
    # outcome you're predicting -- exclude from features, keep for comparison.
    'forecast_quote_cols': ['imf_next_year_growth_forecast_pct'],

    # Columns that are only meaningful (and only ever populated) when another column
    # takes a specific value -- these get imputed with a domain constant, not a
    # statistical average. Map: column -> (condition_col, condition_value, constant).
    'structural_missing_cols': {
        'disaster_damage_pct_gdp': ('natural_disaster_event', 'Yes', 0.0),
    },

    'id_cols': [],
    'cardinality_threshold': 5,
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
}


## 2. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

RANDOM_STATE = CONFIG['random_state']

df_raw = pd.read_csv(CONFIG['data_path'])
df_raw = df_raw.drop(columns=[c for c in CONFIG['id_cols'] if c in df_raw.columns])
print(df_raw.shape)
df_raw.head()


## 3. Generic Cleaning Utilities (reuse as-is)

In [ ]:
def parse_accounting_pct(series):
    """Parse percentage strings using accounting-style negative notation:
    '4.2%' -> 4.2, '(1.7%)' -> -1.7. Also handles a plain '-1.7%' minus-sign form.
    """
    s = series.astype(str).str.strip()
    is_negative = s.str.startswith('(')
    numeric_part = (s.str.replace('(', '', regex=False)
                      .str.replace(')', '', regex=False)
                      .str.replace('%', '', regex=False)
                      .astype(float))
    return np.where(is_negative, -numeric_part.abs(), numeric_part)


def normalize_identifier(series):
    """Strip whitespace, collapse internal whitespace runs, and title-case a
    categorical identifier column (country/company/region names, etc.)."""
    return series.astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()


def audit_dataframe(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(),
        'n_missing': df.isnull().sum(),
        'sample': [df[c].dropna().unique()[:3].tolist() for c in df.columns],
    })

audit = audit_dataframe(df_raw)
print("Duplicate rows:", df_raw.duplicated().sum())
audit


## 4. Cleaning & Feature Engineering (project-specific — EDIT THIS SECTION)

In [ ]:
def check_structural_missingness(df, structural_map):
    """For each configured structural-missing column, verify (via crosstab) that its
    sentinel-missing rows line up with the expected condition column/value."""
    for col, (cond_col, cond_val, _) in structural_map.items():
        if col in df.columns and cond_col in df.columns:
            is_sentinel = df[col].astype(str).isin(['..', 'N/A', 'NA', ''])
            print(f"\n{col} missingness vs. {cond_col}:")
            print(pd.crosstab(df[cond_col], is_sentinel))

check_structural_missingness(df_raw, CONFIG['structural_missing_cols'])


In [ ]:
def clean_raw_columns(df):
    """EDIT ME. Apply parse_accounting_pct() to every percentage-formatted column
    (INCLUDING the target -- check!). Apply normalize_identifier() to any
    country/company/entity-name column. Handle sentinel-missing columns:
    - random reporting gaps: flag column + median impute
    - structural absence (see CONFIG['structural_missing_cols']): impute the
      configured constant, no flag needed
    """
    df = df.copy()
    for col, (cond_col, cond_val, constant) in CONFIG['structural_missing_cols'].items():
        if col in df.columns:
            is_sentinel = df[col].astype(str).isin(['..', 'N/A', 'NA', ''])
            parsed = np.full(len(df), np.nan)
            if (~is_sentinel).any():
                parsed[~is_sentinel] = parse_accounting_pct(df.loc[~is_sentinel, col])
            df[col] = np.nan_to_num(parsed, nan=constant)
    # TODO: dataset-specific unit/percentage/identifier cleaning goes here
    return df


def engineer_domain_features(df):
    """EDIT ME. Ordinal-encode any naturally-ordered categories. Hand-engineer any
    interaction terms domain knowledge suggests, especially SPARSE ones a flexible
    model might not reliably discover on its own (see Background Theory, Section 5).
    """
    df = df.copy()
    # TODO: ordinal maps + interaction features go here
    return df


df = clean_raw_columns(df_raw)
df = engineer_domain_features(df)
df.head()


## 5. EDA (generic — reuse as-is)

In [ ]:
def eda_target_distribution(df, target_col):
    print(f"{target_col} skewness: {df[target_col].skew():.2f}")
    plt.figure(figsize=(7, 4))
    sns.histplot(df[target_col], kde=True)
    plt.axvline(0, color='r', linestyle='--')
    plt.title(f'Distribution of {target_col}')
    plt.show()

eda_target_distribution(df, CONFIG['target_col'])


## 6. The Forecast/Quote Leakage Audit (generic — reuse the mechanics, apply judgment to the decision)

In [ ]:
def leakage_audit(df, target_col, known_cols, corr_threshold=0.7):
    numeric_df = df.select_dtypes(include='number')
    corrs = numeric_df.corr()[target_col].sort_values(ascending=False).drop(target_col)
    suspiciously_high = corrs[corrs.abs() >= corr_threshold]
    print("Top correlations with target:")
    print(corrs.head(10))
    print("\nFlagged for review (|corr| >=", corr_threshold, "):")
    for col, val in suspiciously_high.items():
        flagged = "already in forecast_quote_cols" if col in known_cols else "** REVIEW: not yet excluded **"
        print(f"  {col:35s} corr={val:.3f}  ({flagged})")
    return corrs

_ = leakage_audit(df, CONFIG['target_col'], CONFIG['forecast_quote_cols'])


**Remember:** don't stop at the correlation number. For every flagged column, ask what it *represents* — a settled fact, a third party's own prediction/opinion about the same outcome, or a genuinely independent signal — before deciding whether to exclude it (see Background Theory, Section 6, for why a moderate correlation can still mean strong leakage, and vice versa).

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(df, target_col, exclude_cols):
    numeric_df = df.select_dtypes(include='number')
    cols = [c for c in numeric_df.columns if c not in [target_col] + exclude_cols]
    X_vif = numeric_df[cols].dropna()
    return pd.DataFrame({
        'feature': X_vif.columns,
        'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    }).sort_values('VIF', ascending=False)

vif_table = compute_vif(df, CONFIG['target_col'], exclude_cols=CONFIG['forecast_quote_cols'])
vif_table.head(10)


## 7. Leakage-Safe Encoding (generic — reuse as-is)

In [ ]:
def split_columns_by_cardinality(df, threshold, exclude_cols):
    cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in exclude_cols]
    low_card = [c for c in cat_cols if df[c].nunique() < threshold]
    high_card = [c for c in cat_cols if df[c].nunique() >= threshold]
    return low_card, high_card


def prepare_features(df, config, ordinal_originals=()):
    exclude_from_cat = [config['target_col']] + config['forecast_quote_cols'] + list(ordinal_originals)
    low_card, high_card = split_columns_by_cardinality(df, config['cardinality_threshold'], exclude_from_cat)

    df_enc = df.drop(columns=[c for c in ordinal_originals if c in df.columns])
    df_enc = pd.get_dummies(df_enc, columns=low_card, drop_first=True)
    bool_cols = df_enc.select_dtypes(include='bool').columns
    df_enc[bool_cols] = df_enc[bool_cols].astype(int)

    drop_cols = [config['target_col']] + config['forecast_quote_cols']
    X = df_enc.drop(columns=[c for c in drop_cols if c in df_enc.columns])
    y = df_enc[config['target_col']]
    return X, y, low_card, high_card


from sklearn.model_selection import train_test_split

# EDIT: pass the names of any columns you ordinal-encoded in engineer_domain_features()
ORDINAL_ORIGINALS = []

X, y, low_card, high_card = prepare_features(df, CONFIG, ordinal_originals=ORDINAL_ORIGINALS)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'], random_state=RANDOM_STATE
)

target_encoding_maps = {}
global_mean = y_train.mean()
for col in high_card:
    means = y_train.groupby(X_train[col]).mean()
    target_encoding_maps[col] = means
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(global_mean)

print("Nulls:", X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
X_train.shape, X_test.shape


## 8. Model Zoo, With and Without Any Hand-Engineered Interaction (generic — reuse as-is)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"{name:35s} MAE={mae:10.4f}  RMSE={rmse:10.4f}  R2={r2:.3f}")

y_pred_baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
evaluate('Mean baseline', y_test, y_pred_baseline)

lin = LinearRegression().fit(X_train, y_train)
evaluate('Linear Regression', y_test, lin.predict(X_test))

ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate('Ridge', y_test, ridge.predict(X_test))

rf = RandomForestRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate('Random Forest', y_test, rf.predict(X_test))

gb = GradientBoostingRegressor(random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate('Gradient Boosting', y_test, gb.predict(X_test))

fitted_models = {'Linear Regression': lin, 'Ridge': ridge, 'Random Forest': rf, 'Gradient Boosting': gb}
pd.DataFrame(results).sort_values('MAE')


**Reminder:** if you engineered a sparse interaction term, explicitly compare a linear model with vs. without it — the improvement (however modest) is itself evidence about whether the interaction is real and worth keeping.

## 9. Cross-Validation & Hyperparameter Tuning (generic — edit the grid)

In [ ]:
from sklearn.model_selection import cross_val_score, RandomizedSearchCV

candidate_name = min(results[1:], key=lambda r: r['MAE'])['model']
candidate = fitted_models[candidate_name]
print("Tuning:", candidate_name)

cv_scores = -cross_val_score(candidate, X_train, y_train, cv=CONFIG['cv_folds'],
                              scoring='neg_mean_absolute_error')
print(f"{CONFIG['cv_folds']}-fold CV MAE: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

param_dist = {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]} if hasattr(candidate, 'alpha') else {
    'n_estimators': [100, 200, 400], 'max_depth': [None, 5, 10, 20]
}
search = RandomizedSearchCV(candidate.__class__(random_state=RANDOM_STATE), param_distributions=param_dist,
                             n_iter=10, cv=CONFIG['cv_folds'], scoring='neg_mean_absolute_error',
                             random_state=RANDOM_STATE, n_jobs=-1)
search.fit(X_train, y_train)
final_model = search.best_estimator_
evaluate('Final (tuned)', y_test, final_model.predict(X_test))


## 10. Diagnostics & Feature Importance (generic — reuse as-is)

In [ ]:
def plot_diagnostics(y_test, y_pred):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(y_test, y_pred, alpha=0.5)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[0].plot(lims, lims, 'r--'); axes[0].set_title('Predicted vs Actual')
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.5); axes[1].axhline(0, color='r', linestyle='--')
    axes[1].set_title('Residuals vs Predicted')
    plt.tight_layout(); plt.show()

y_pred_final = final_model.predict(X_test)
plot_diagnostics(y_test, y_pred_final)


In [ ]:
from sklearn.inspection import permutation_importance

def plot_feature_importance(model, X_train, X_test, y_test, top_n=10):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        idx = np.argsort(importances)[-top_n:][::-1]
        sns.barplot(x=importances[idx], y=X_train.columns[idx], ax=axes[0])
        axes[0].set_title('Impurity-based importance')
    elif hasattr(model, 'coef_'):
        coefs = pd.Series(model.coef_, index=X_train.columns).sort_values(key=abs, ascending=False)
        sns.barplot(x=coefs.head(top_n).values, y=coefs.head(top_n).index, ax=axes[0])
        axes[0].set_title('|Coefficient| ranking')

    perm = permutation_importance(model, X_test, y_test, n_repeats=10,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    pidx = perm.importances_mean.argsort()[-top_n:][::-1]
    sns.barplot(x=perm.importances_mean[pidx], y=X_test.columns[pidx], ax=axes[1])
    axes[1].set_title('Permutation importance')
    plt.tight_layout(); plt.show()

plot_feature_importance(final_model, X_train, X_test, y_test)


## 11. Scenario Simulation (generic — adapt the interaction-recompute step)

In [ ]:
def simulate_scenario(model, base_row, changes: dict, interaction_recompute=None):
    """interaction_recompute(row) -> row, an optional function that recomputes any
    interaction columns after `changes` are applied (see Section 4's hand-engineered
    interaction, if any)."""
    baseline_pred = model.predict(base_row)[0]
    scenario_row = base_row.copy()
    for col, val in changes.items():
        scenario_row[col] = val
    if interaction_recompute is not None:
        scenario_row = interaction_recompute(scenario_row)
    scenario_pred = model.predict(scenario_row)[0]
    return baseline_pred, scenario_pred, scenario_pred - baseline_pred

# Example:
# base_row = X_test.loc[[some_index]]
# baseline, scenario, delta = simulate_scenario(
#     final_model, base_row, {'some_driver_column': new_value}
# )
# print(f"Baseline: {baseline:.2f}  Scenario: {scenario:.2f}  Impact: {delta:+.2f}")


**Caution:** only trust scenario predictions that stay within (or close to) the range of input combinations seen in training. A model asked to predict for a combination of inputs far outside anything it was trained on is extrapolating, not interpolating — treat those results as much less reliable.

## 12. Persistence & Generic Inference Wrapper

In [ ]:
import joblib

def save_artifacts(model, path_prefix='model'):
    joblib.dump(model, f'{path_prefix}.pkl')
    joblib.dump({
        'target_encoding_maps': target_encoding_maps,
        'global_mean': global_mean,
        'model_columns': list(X_train.columns),
        'low_card_cols': low_card,
        'high_card_cols': high_card,
        'config': CONFIG,
    }, f'{path_prefix}_encoders.pkl')

def predict_from_raw(raw_dict, model, path_prefix='model'):
    art = joblib.load(f'{path_prefix}_encoders.pkl')
    row = pd.DataFrame([raw_dict])
    row = clean_raw_columns(row)
    row = engineer_domain_features(row)
    row = row.drop(columns=[c for c in ORDINAL_ORIGINALS if c in row.columns])
    row = pd.get_dummies(row, columns=art['low_card_cols'], drop_first=True)
    bool_cols = row.select_dtypes(include='bool').columns
    row[bool_cols] = row[bool_cols].astype(int)
    for col in art['high_card_cols']:
        if col in row.columns:
            row[col] = row[col].map(art['target_encoding_maps'][col]).fillna(art['global_mean'])
    row = row.reindex(columns=art['model_columns'], fill_value=0)
    return float(model.predict(row)[0])

save_artifacts(final_model)
print("Artifacts saved.")


## 13. Per-Project Checklist (quick reference)

- [ ] Update `CONFIG`, including `forecast_quote_cols` and `structural_missing_cols`
- [ ] Check every percentage/currency column for accounting-style parentheses-negative formatting — **including the target**
- [ ] Normalize any entity-name (country/company) column for whitespace and casing
- [ ] Cross-tabulate any suspected structural-missingness column against its likely condition column before choosing an imputation strategy
- [ ] Fill in `engineer_domain_features()`, hand-engineering any sparse-but-plausible interaction
- [ ] Run the leakage audit and reason about BOTH correlation strength and what each flagged column represents
- [ ] Confirm zero nulls in `X_train`/`X_test` after encoding
- [ ] Compare linear (with/without interaction), Ridge, and tree-ensemble models — don't assume which wins
- [ ] Cross-validate before trusting a single split
- [ ] If scenario simulation is part of the deliverable, caution against extrapolating far outside the training data's range
- [ ] Persist encoders alongside the model
